# Our PR2 MJCF in the Multiverse apartment

This notebook uses repository-local PR2, apartment, milk-box, and cereal-box MJCF resources. It adapts the parsed PR2 planar base to an SDT `OmniDrive`, then executes the standard PyCRAM park-arms, torso, and navigation actions through Giskard while `MujocoSynchronizer` mirrors the resulting world state into MuJoCo.

The scene starts with parked arms, a low torso, and closed kitchen drawers. The free milk and cereal boxes begin above the right countertop and visibly fall onto it under MuJoCo gravity and contacts.


In [ ]:
import os
import time
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

import mujoco
import numpy as np

from physics_simulators.mujoco_simulator import MujocoSimulator
from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import Arms
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from semantic_digital_twin.adapters.mjcf import MJCFParser
from semantic_digital_twin.adapters.multi_sim import MujocoBuilder, MujocoSynchronizer
from semantic_digital_twin.callbacks.callback import StateChangeCallback
from semantic_digital_twin.datastructures.definitions import StaticJointState, TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Cereal, Milk
from semantic_digital_twin.spatial_types.spatial_types import HomogeneousTransformationMatrix, Pose
from semantic_digital_twin.world_description.connections import (
    ActiveConnection1DOF,
    FixedConnection,
    OmniDrive,
)
from semantic_digital_twin.world_description.geometry import Box, Color, Scale
from semantic_digital_twin.world_description.shape_collection import ShapeCollection


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'semantic_digital_twin/resources/mujoco_resources').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


REPO_ROOT = find_repo_root()
MUJOCO_RESOURCES = REPO_ROOT / 'semantic_digital_twin/resources/mujoco_resources'
PR2_MJCF = (MUJOCO_RESOURCES / 'pr2/pr2.xml').resolve()
APARTMENT_MJCF = (MUJOCO_RESOURCES / 'apartment/apartment.xml').resolve()
MILK_MJCF = (MUJOCO_RESOURCES / 'objects/milk_box/milk_box.xml').resolve()
CEREAL_MJCF = (MUJOCO_RESOURCES / 'objects/cereal_box/cereal_box.xml').resolve()
GENERATED_SCENE = Path('/tmp/pr2_multiverse_apartment.xml')
HEADLESS = os.environ.get('MUJOCO_HEADLESS', '0') == '1'

# Clear aisle between the wall cabinets and the kitchen island.
PR2_X = 1.2
PR2_Y = 3.7
PR2_YAW = -np.pi / 2

for resource in (PR2_MJCF, APARTMENT_MJCF, MILK_MJCF, CEREAL_MJCF):
    if not resource.is_file():
        raise FileNotFoundError(resource)
print(f'PR2:       {PR2_MJCF}')
print(f'Apartment: {APARTMENT_MJCF}')
print(f'Milk:      {MILK_MJCF}')
print(f'Cereal:    {CEREAL_MJCF}')
print(f'Output:    {GENERATED_SCENE}')
print(f'Headless:  {HEADLESS}')


## Parse and prepare the SDT world

The generated PR2 MJCF represents planar motion as three serial joints. Before semantic initialization, this cell replaces that chain with the SDT `OmniDrive` expected by PR2 navigation. It also converts NumPy scalar limits from the MJCF parser to native Python floats so Giskard can combine them with symbolic variables.


In [ ]:
pr2_world = MJCFParser(str(PR2_MJCF)).parse()

apartment_parser = MJCFParser(str(APARTMENT_MJCF))
# Each parser otherwise creates a root body named 'world'.
apartment_parser.spec.worldbody.name = 'apartment_world'
apartment_world = apartment_parser.parse()

pr2_world.merge_world(
    apartment_world,
    root_connection=FixedConnection(
        parent=pr2_world.root, child=apartment_world.root
    ),
)
world = pr2_world

# Start visibly above the countertop so viewer startup demonstrates gravity.
OBJECT_INITIAL_POSES = {
    'milk_box': (3.0, 1.70, 1.70),
    'cereal_box': (3.0, 2.05, 1.72),
}


def merge_free_mjcf_object(mjcf_path, world_root_name, body_name, pose):
    parser = MJCFParser(str(mjcf_path))
    parser.spec.worldbody.name = world_root_name
    object_world = parser.parse()
    object_world_root = object_world.root
    world.merge_world(
        object_world,
        root_connection=FixedConnection(parent=world.root, child=object_world_root),
    )
    object_body = world.get_body_by_name(body_name)
    with world.modify_world():
        world.move_branch(object_body, world.root)
    object_body.parent_connection.origin = pose
    return object_body


milk_body = merge_free_mjcf_object(
    MILK_MJCF,
    'milk_world',
    'milk_box',
    HomogeneousTransformationMatrix.from_xyz_rpy(
        *OBJECT_INITIAL_POSES['milk_box'], reference_frame=world.root
    ),
)
cereal_body = merge_free_mjcf_object(
    CEREAL_MJCF,
    'cereal_world',
    'cereal_box',
    HomogeneousTransformationMatrix.from_xyz_rpy(
        *OBJECT_INITIAL_POSES['cereal_box'], reference_frame=world.root
    ),
)


def normalize_dof_limits():
    for dof in world.degrees_of_freedom:
        for limits in (dof.limits.lower, dof.limits.upper):
            for derivative in ('position', 'velocity', 'acceleration', 'jerk'):
                value = getattr(limits, derivative)
                if value is not None:
                    setattr(limits, derivative, float(value))


def replace_planar_base_with_omnidrive():
    world_root = world.root
    base_footprint = world.get_body_by_name('base_footprint')
    old_connections = [
        world.get_connection_by_name(name)
        for name in ('base_x_joint', 'base_y_joint', 'base_yaw_joint')
    ]
    old_dofs = {dof for connection in old_connections for dof in connection.dofs}

    with world.modify_world():
        for actuator in list(world.actuators):
            if any(dof in old_dofs for dof in actuator.dofs):
                world.remove_actuator(actuator)
        for connection in old_connections:
            world.remove_connection(connection)
        world.remove_kinematic_structure_entity(world.get_body_by_name('pr2_planar_y'))
        world.remove_kinematic_structure_entity(world.get_body_by_name('pr2_planar_x'))
        world.delete_orphaned_dofs()

        drive = OmniDrive.create_with_dofs(
            world=world,
            parent=world_root,
            child=base_footprint,
            name=PrefixedName('odom_combined_T_base_footprint'),
        )
        world.add_connection(drive)
        drive.origin = HomogeneousTransformationMatrix.from_xyz_rpy(
            x=PR2_X,
            y=PR2_Y,
            yaw=PR2_YAW,
            reference_frame=world_root,
        )
    return drive


normalize_dof_limits()
drive = replace_planar_base_with_omnidrive()
print(f'{len(world.bodies)} bodies')
print(f'{len(world.connections)} connections')
print(f'{len(world.actuators)} parsed joint actuators')


## Add PR2 and object semantics

The generated PR2 MJCF omits decorative SRDF-only links, so its semantic annotations are initialized without loading the collision-ignore SRDF. A conservative invisible base collision proxy supplies the footprint required by the standard navigation precondition; MuJoCo still uses the original model geoms for physics.


In [ ]:
with world.modify_world():
    pr2 = PR2._init_empty_robot(world)
    pr2._setup_semantic_annotations()
    pr2._setup_velocity_limits()
    pr2._setup_hardware_interfaces()
    pr2._setup_joint_states()
    world.add_semantic_annotation(pr2)

    milk = Milk(root=milk_body)
    cereal = Cereal(root=cereal_body)
    world.add_semantic_annotation(milk)
    world.add_semantic_annotation(cereal)

    # The source MJCF has visual meshes but no collision shape on base_link.
    base_link = world.get_body_by_name('base_link')
    base_link.collision = ShapeCollection(
        shapes=[
            Box(
                scale=Scale(0.68, 0.68, 0.25),
                origin=HomogeneousTransformationMatrix.from_xyz_rpy(
                    z=0.12, reference_frame=base_link
                ),
                color=Color(0.0, 0.0, 0.0, 0.0),
            )
        ],
        reference_frame=base_link,
    )

context = Context(world=world, robot=pr2)
context.evaluate_conditions = True
assert pr2.drive is drive and pr2.drive.has_hardware_interface
print(f'Objects: {milk.root.name.name}, {cereal.root.name.name}')
print(f'Controlled drive: {pr2.drive.name.name}')


## Initial PR2 and apartment state

The `OmniDrive` starts in the clear kitchen aisle. Both arms use the PR2 semantic parked state, the torso starts low, and every active drawer or door joint starts at its zero-valued closed position.


In [ ]:
initial_joint_targets = {}
for arm in pr2.arms:
    parked = arm.get_joint_state_by_type(StaticJointState.PARK)
    initial_joint_targets.update(
        {connection.name.name: value for connection, value in parked.items()}
    )
initial_joint_targets.update(
    {
        connection.name.name: value
        for connection, value in pr2.torso.get_joint_state_by_type(TorsoState.LOW).items()
    }
)

# The source kitchen has zero as the closed position for every drawer and door.
closed_apartment_joints = [
    connection.name.name
    for connection in world.connections
    if isinstance(connection, ActiveConnection1DOF)
    and ('drawer' in connection.name.name or 'door' in connection.name.name)
]
initial_joint_targets.update({name: 0.0 for name in closed_apartment_joints})
for name, value in initial_joint_targets.items():
    world.get_connection_by_name(name).position = value
world.notify_state_change()
print(f'Initialized {len(closed_apartment_joints)} drawer/door joints as closed.')


## Build one MJCF scene and add lighting

The builder emits the SDT `OmniDrive` as x, y, and yaw MuJoCo joints. XML post-processing adds position actuators for those generated joints, plus the headlight, skybox, directional lights, gravity, and an implicit integrator.


In [ ]:
class BalancedMujocoBuilder(MujocoBuilder):
    def _start_build(self, file_path):
        super()._start_build(file_path)
        self.spec.compiler.balanceinertia = True
        self.spec.compiler.boundmass = 1e-6
        self.spec.compiler.boundinertia = 1e-6


BalancedMujocoBuilder().build_world(world, str(GENERATED_SCENE))

tree = ET.parse(GENERATED_SCENE)
root = tree.getroot()
compiler = root.find('compiler')
compiler.set('balanceinertia', 'true')
compiler.set('boundmass', '0.000001')
compiler.set('boundinertia', '0.000001')

option = root.find('option')
if option is None:
    option = ET.Element('option')
    root.insert(1, option)
option.set('gravity', '0 0 -9.81')
option.set('integrator', 'implicitfast')

asset = root.find('asset')
visual = root.find('visual')
if visual is None:
    visual = ET.Element('visual')
    root.insert(list(root).index(asset), visual)
ET.SubElement(
    visual,
    'headlight',
    diffuse='0.7 0.7 0.7',
    ambient='0.35 0.35 0.35',
    specular='0.1 0.1 0.1',
)
ET.SubElement(visual, 'rgba', haze='0.15 0.25 0.35 1')
ET.SubElement(visual, 'global', azimuth='120', elevation='-20')
ET.SubElement(
    asset,
    'texture',
    name='scene_skybox',
    type='skybox',
    builtin='gradient',
    rgb1='0.3 0.5 0.7',
    rgb2='0 0 0',
    width='512',
    height='3072',
)

worldbody = root.find('worldbody')
ET.SubElement(
    worldbody,
    'light',
    name='ceiling_light',
    pos='0 0 6',
    dir='0 0 -1',
    directional='true',
    diffuse='0.8 0.8 0.8',
    specular='0.2 0.2 0.2',
)
ET.SubElement(
    worldbody,
    'light',
    name='fill_light',
    pos='2 -2 4',
    dir='-0.3 0.3 -1',
    directional='true',
    diffuse='0.45 0.45 0.45',
    specular='0.1 0.1 0.1',
    castshadow='false',
)

actuator = root.find('actuator')
if actuator is None:
    actuator = ET.SubElement(root, 'actuator')
for name, suffix, kp, ctrlrange in (
    ('base_x_motor', 'x', '10000', '-100 100'),
    ('base_y_motor', 'y', '10000', '-100 100'),
    ('base_yaw_motor', 'yaw', '5000', '-25.1327412287 25.1327412287'),
):
    ET.SubElement(
        actuator,
        'position',
        name=name,
        joint=f'{drive.name.name}_{suffix}',
        kp=kp,
        ctrllimited='true',
        ctrlrange=ctrlrange,
        forcelimited='true',
        forcerange='-1000000 1000000',
    )

# Keep passive apartment furniture closed while gravity and contacts are active.
for joint_name in closed_apartment_joints:
    ET.SubElement(
        actuator,
        'position',
        name=f'{joint_name}_closed_motor',
        joint=joint_name,
        kp='1000',
        forcelimited='true',
        forcerange='-100000 100000',
    )

ET.indent(tree, space='  ')
tree.write(GENERATED_SCENE, encoding='utf-8', xml_declaration=True)
model = mujoco.MjModel.from_xml_path(str(GENERATED_SCENE))
print(
    f'Compiled: {model.nbody} bodies, {model.njnt} joints, '
    f'{model.nu} actuators, {model.ngeom} geoms, {model.nlight} lights'
)


## Launch the viewer and demonstrate gravity

This starts MuJoCo with gravity and contacts enabled. The milk and cereal boxes begin above the counter; the startup loop advances physics and refreshes the viewer in real time so their fall is visible. A state-change callback then holds position actuators and refreshes the viewer as Giskard updates the SDT world.


In [ ]:
simulator = MujocoSimulator(
    file_path=str(GENERATED_SCENE),
    _headless=HEADLESS,
    _step_size=0.001,
    config={
        'gravity': True,
        'contact': True,
        'integrator': mujoco.mjtIntegrator.mjINT_IMPLICITFAST,
    },
)
simulator.set_joints_values(initial_joint_targets)

drive_joint_targets = {
    f'{drive.name.name}_x': PR2_X,
    f'{drive.name.name}_y': PR2_Y,
    f'{drive.name.name}_yaw': PR2_YAW,
}
simulator.set_joints_values(drive_joint_targets)

# Free joints are emitted before the articulated joints in MuJoCo qpos, while
# the merged-world keyframe is serialized in world order. Initialize them by
# joint address before replacing that stale generated keyframe with `home`.
for body_name, xyz in OBJECT_INITIAL_POSES.items():
    body_id = mujoco.mj_name2id(
        simulator._mj_model, mujoco.mjtObj.mjOBJ_BODY, body_name
    )
    joint_id = int(simulator._mj_model.body_jntadr[body_id])
    qpos_address = int(simulator._mj_model.jnt_qposadr[joint_id])
    simulator._mj_data.qpos[qpos_address:qpos_address + 7] = (
        *xyz,
        1.0,
        0.0,
        0.0,
        0.0,
    )
mujoco.mj_forward(simulator._mj_model, simulator._mj_data)


def hold_position_actuators_at_qpos():
    model = simulator._mj_model
    data = simulator._mj_data
    for actuator_index in range(model.nu):
        if model.actuator_trntype[actuator_index] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = int(model.actuator_trnid[actuator_index, 0])
        if joint_id < 0:
            continue
        qpos_address = int(model.jnt_qposadr[joint_id])
        data.ctrl[actuator_index] = data.qpos[qpos_address]


hold_position_actuators_at_qpos()
simulator.save(key_name='home')
synchronizer = MujocoSynchronizer(_world=world, simulator=simulator)
simulator.start(simulate_in_thread=False)

# Advance two seconds of physics. In the interactive viewer this is paced
# close to real time so the boxes can be seen falling onto the counter.
GRAVITY_DEMO_SECONDS = 2.0
SIMULATION_STEP_SIZE = 0.001
RENDER_EVERY_STEPS = 10
gravity_demo_steps = int(GRAVITY_DEMO_SECONDS / SIMULATION_STEP_SIZE)
for step in range(gravity_demo_steps):
    simulator.step()
    if step % RENDER_EVERY_STEPS == 0:
        simulator.renderer.sync()
        if not simulator.headless:
            time.sleep(SIMULATION_STEP_SIZE * RENDER_EVERY_STEPS)

# Contact can nudge a passive furniture joint during the gravity demo. Restore
# the deterministic closed scene once, then let the closure actuators hold it.
simulator.set_joints_values({name: 0.0 for name in closed_apartment_joints})
hold_position_actuators_at_qpos()
mujoco.mj_forward(simulator._mj_model, simulator._mj_data)
simulator.renderer.sync()
synchronizer._last_sync_time = 0.0
synchronizer._sim_to_world()


@dataclass(eq=False)
class MujocoControlAndRenderCallback(StateChangeCallback):
    simulator: MujocoSimulator
    control_period: float = 1.0 / 50.0

    def _notify(self, **kwargs):
        hold_position_actuators_at_qpos()
        mujoco.mj_forward(self.simulator._mj_model, self.simulator._mj_data)
        self.simulator.renderer.sync()
        if not self.simulator.headless:
            time.sleep(self.control_period)
        self.update_previous_world_state()


mujoco_control_callback = MujocoControlAndRenderCallback(
    _world=world,
    simulator=simulator,
)
print('PR2 apartment scene started; milk and cereal fell onto the counter under gravity.')


## Standard PyCRAM action plan

No notebook-local action descriptions are needed. The plan below uses the same stock actions used by PyCRAM task plans, with condition evaluation enabled.


In [ ]:
# With yaw=-pi/2, reducing y moves the robot forward through the clear aisle.
NAVIGATION_GOAL = Pose.from_xyz_rpy(
    x=1.2,
    y=3.2,
    yaw=-np.pi / 2,
    reference_frame=world.root,
)
simple_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        NavigateAction(NAVIGATION_GOAL, keep_joint_states=True),
    ],
    context=context,
).plan
print('Plan: park arms -> raise torso -> move forward -> stop')


## Execute the plan

Run this cell when you are ready to execute the standard PyCRAM actions through Giskard. `MujocoSynchronizer` copies each resulting SDT state update into MuJoCo.


In [ ]:
with simulated_robot:
    simple_plan.perform()

drive_state = world.state
print(
    'Plan finished:',
    f"torso={world.get_connection_by_name('torso_lift_joint').position:.3f}",
    f"base=({drive_state[pr2.drive.x.id].position:.3f}, "
    f"{drive_state[pr2.drive.y.id].position:.3f}, "
    f"yaw={drive_state[pr2.drive.yaw.id].position:.3f})",
)


## Stop before rerunning


In [ ]:
if 'mujoco_control_callback' in globals():
    mujoco_control_callback.stop()
if 'synchronizer' in globals():
    synchronizer.stop()
if 'simulator' in globals():
    simulator.stop()
print('Scene stopped.')
